# Method E — ACH arbiter over Method D's candidates (frontier LLM)

**Question this notebook answers:** does reasoning from a frontier LLM, applied
*on top of* the production retrieve+rerank pipeline, improve on Method D alone — and
does it do so in a way that's actually likely to help, informed by external research
rather than a repeat of the earlier bake-off's naive LLM methods?

## Why not just replay Method B or C with a stronger model

The original `classification_bakeoff.ipynb` found Method D (retrieve + cross-encoder
rerank, zero LLM) beat Method B (LLM-only, full 79-technique catalogue) and roughly
tied Method C (hybrid: retrieval narrows candidates, LLM ranks among them) — using a
small **local** model (`qwen2.5:7b`), explicitly caveated there as "a floor, not a
ceiling" for LLM-based methods.

Before assuming a frontier model changes that, this design was checked against outside
evidence:

- A 2026 arXiv study evaluating 7 open-source LLMs (8B–236B params) on ATT&CK technique
  classification found **naive zero/few-shot classification over the full catalogue
  tops out at micro-F1 0.22 even at 236B params** — chain-of-thought reasoning gave
  only +0.02–0.05, few-shot ≤+0.01. **Retrieval grounding (RAG) was the one lever that
  mattered, raising F1 to 0.32.** Their conclusion: "retrieval grounding, not prompt
  tuning, is the productive direction."
- This directly implies a frontier model replaying **Method B's shape** (rank all 79
  techniques cold) is unlikely to be the right design, regardless of model strength.

**So Method E is not Method B or C replayed with a bigger model.** It's a new design:
the frontier model reasons only over Method D's already-retrieved, already-reranked
top-K candidates — the same "retrieval grounds, verification decides" shape that won
the original bake-off, but with **Analysis of Competing Hypotheses (ACH)** — the
CIA/intelligence-community structured-analytic-technique for reasoning under
uncertainty (Heuer, *Psychology of Intelligence Analysis*) — as the verification step
instead of a cross-encoder score. ACH's core discipline: weigh evidence *for and
against* each hypothesis, and favour the one with the **least disconfirming evidence**,
not simply the one that sounds most plausible on first read. `research/lib/ach_prompts.py`
implements the prompt; see its docstring for the full reasoning.

## Design notes

- One `opencode` call **per example**, not per candidate — all K candidates go in one
  prompt so the model compares them in context, and per-call CLI overhead (each
  `opencode run` invocation loads real context, ~7k input tokens even for a one-word
  ping — confirmed empirically before this notebook was written) isn't paid K times
  over.
- `valid_ids` passed to `opencode_client.classify()` is **the K-candidate set for that
  example, not all 79 techniques** — a technique ID outside the offered candidates
  counts as "hallucinated" in this bookkeeping even if it's a real ATT&CK ID, since the
  arbiter was explicitly scoped to reason only about the given candidates. This is a
  deliberate semantic narrowing from how Methods B/C used the same field.
- Scored with the exact same `lib/metrics.rank_score_method` used throughout the bake-
  off — MRR/MAP/Recall@k, no forced top-k cutoff, no new scoring code to separately
  trust.
- Method D is **recomputed fresh on this notebook's own sample** (not pulled from the
  original bake-off's numbers, which ran on a different stratified draw) so the E-vs-D
  comparison here is a same-sample, apples-to-apples one.

## 1. Setup & environment check

In [1]:
import sys
import time
from collections import Counter
from pathlib import Path

import pandas as pd


def _find_research_dir() -> Path:
    here = Path.cwd()
    for candidate in (here, here / "research", here.parent):
        if (candidate / "lib" / "attack_data.py").exists():
            return candidate
    raise RuntimeError("Could not locate the research/ directory - run from research/ or repo root.")


RESEARCH_DIR = _find_research_dir()
REPO_ROOT = RESEARCH_DIR.parent
DATA_DIR = RESEARCH_DIR / "data"
sys.path.insert(0, str(RESEARCH_DIR))
sys.path.insert(0, str(REPO_ROOT))

from lib import ach_prompts, eval_data, metrics, ollama_client, opencode_client  # noqa: E402

from app.classification.corpus import get_technique_by_id  # noqa: E402
from app.classification.reranking import rerank  # noqa: E402
from app.classification.retrieval import HYBRID_RETRIEVAL_N, retrieve  # noqa: E402

# --- Knobs ---
FRONTIER_MODEL = "opencode-go/qwen3.8-max"  # tier-3 rotation model, confirmed reachable
                                             # via ping() before this notebook was written
ARBITER_K = 6            # candidates shown to the arbiter per example
PILOT_N = 20             # sanity-check sample before spending on the full run
EVAL_N = 50              # stratified sample size for the real E-vs-D comparison
SAMPLE_SEED = 42         # matches the bake-off's seed for consistency across notebooks

# Dedicated cache file, not the bake-off's shared .llm_cache.json -- keeps this
# notebook's entries isolated so a bad run here can never poison (or be poisoned by)
# the original bake-off's cached Ollama results.
cache = ollama_client.ResponseCache(DATA_DIR / ".llm_cache_frontier_arbiter.json")

print(f"research dir  : {RESEARCH_DIR}")
print(f"repo root     : {REPO_ROOT}")
print(f"frontier model: {FRONTIER_MODEL}   ARBITER_K: {ARBITER_K}")
print(f"HYBRID_RETRIEVAL_N (from app.classification.retrieval): {HYBRID_RETRIEVAL_N}")
print(f"cache entries : {len(cache)} (path: {cache.path})")

C:\Users\cayen_khgz1m4\OneDrive\Desktop\Personal projects\CTI-automation-mvp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


research dir  : C:\Users\cayen_khgz1m4\OneDrive\Desktop\Personal projects\CTI-automation-mvp\research
repo root     : C:\Users\cayen_khgz1m4\OneDrive\Desktop\Personal projects\CTI-automation-mvp
frontier model: opencode-go/qwen3.8-max   ARBITER_K: 6
HYBRID_RETRIEVAL_N (from app.classification.retrieval): 50
cache entries : 40 (path: C:\Users\cayen_khgz1m4\OneDrive\Desktop\Personal projects\CTI-automation-mvp\research\data\.llm_cache_frontier_arbiter.json)


In [2]:
reply = opencode_client.ping(FRONTIER_MODEL)
print(f"Ping {FRONTIER_MODEL!r} -> {reply!r}")
assert reply, "Empty response - is the opencode CLI / opencode-go plan reachable?"
print("Environment OK.")

Ping 'opencode-go/qwen3.8-max' -> 'ready'
Environment OK.


## 2. Load the technique corpus and evaluation set

Reuses the exact same corpus loader and eval harness as every other notebook in this
directory (`app.classification.corpus`, `lib.eval_data`) — no new data-loading code.

In [3]:
technique_by_id = get_technique_by_id()
valid_ids = set(technique_by_id)

examples = eval_data.load_eval_set(
    DATA_DIR / "procedure_examples.jsonl",
    DATA_DIR / "labeled_dataset.jsonl",
    DATA_DIR / "labeled_dataset.local.jsonl",
)
eval_data.validate_technique_ids(examples, valid_ids)
examples = eval_data.compute_name_absent(examples, technique_by_id)

corpus_texts = [f"{t.name}. {t.description}" for t in technique_by_id.values()]
eval_data.assert_no_leakage(corpus_texts, examples)

print(f"Loaded {len(examples)} eval examples (validated, leakage-checked).")
print("By slice:", dict(Counter(e.slice for e in examples)))

Loaded 259 eval examples (validated, leakage-checked).
By slice: {'procedure': 238, 'synthetic-easy': 10, 'synthetic-hard': 6, 'real': 5}


## 3. Method D candidates for the sample (retrieval + rerank, production pipeline)

Calls `app.classification.retrieval.retrieve` and `app.classification.reranking.rerank`
directly — the actual shipped functions, not a notebook-local reimplementation — so
Method E is measurably "arbitration on top of exactly what's in production," and
Method D's own score here is the real production pipeline's score, not a proxy.

In [4]:
def method_d_candidates(text: str, top_n: int = ARBITER_K):
    # Retrieve -> rerank, return the top `top_n` as (id, name, description_full).
    raw = retrieve(text, top_n=HYBRID_RETRIEVAL_N)
    reranked = rerank(text, raw)[:top_n]
    return [
        (c.technique_id, technique_by_id[c.technique_id].name, technique_by_id[c.technique_id].description_full)
        for c in reranked
    ]


def method_d_result(report_id: str, true_ids: list[str], text: str, top_n: int = ARBITER_K):
    cands = method_d_candidates(text, top_n=top_n)
    return metrics.MethodResult(report_id=report_id, true_ids=true_ids, ranked_ids=[c[0] for c in cands])

## 4. Pilot run (n=20) — sanity-check prompt shape and parsing before scaling

In [5]:
pilot = eval_data.sample_by_technique(examples, PILOT_N, seed=SAMPLE_SEED)
print(f"Pilot sample: {len(pilot)} examples covering "
      f"{len({tid for ex in pilot for tid in ex.true_technique_ids})} techniques.")

pilot_results = []
pilot_preds = []
t0 = time.time()
for i, ex in enumerate(pilot, 1):
    candidates = method_d_candidates(ex.behavior_text)
    candidate_ids = {c[0] for c in candidates}
    prompt = ach_prompts.build_ach_prompt(ex.behavior_text, candidates)
    pred = opencode_client.classify(FRONTIER_MODEL, prompt, candidate_ids, cache=cache, timeout=180)
    pilot_preds.append(pred)
    pilot_results.append(
        metrics.MethodResult(
            report_id=ex.id, true_ids=ex.true_technique_ids, ranked_ids=pred.ranked_ids,
            parse_failure=pred.parse_failure, hallucinated_id_count=len(pred.hallucinated),
        )
    )
    print(f"  [{i}/{len(pilot)}] {ex.id}  parsed={not pred.parse_failure}  "
          f"ranked={pred.ranked_ids[:3]}...  ({time.time()-t0:.0f}s elapsed)")

pf = sum(r.parse_failure for r in pilot_results)
print(f"\nPilot done. parse failures {pf}/{len(pilot_results)}  ({time.time()-t0:.0f}s total)")
assert pf < len(pilot_results), "Every pilot call failed to parse - inspect the prompt/model before scaling."

# Spot-check the first pilot item's raw reply for the ACH fields actually showing up
# (evidence_for/evidence_against/verdict) - reuses the call already made above, no extra cost.
print("\nWorked example raw reply (first pilot item, truncated to 800 chars):")
print(pilot_preds[0].raw_text[:800])

Pilot sample: 20 examples covering 21 techniques.


  [1/20] proc-0106  parsed=True  ranked=['T0862', 'T0863', 'T0817']...  (30s elapsed)


  [2/20] proc-0104  parsed=True  ranked=['T0883', 'T0819', 'T0866']...  (33s elapsed)


  [3/20] proc-0149  parsed=True  ranked=['T0809', 'T0826', 'T0872']...  (36s elapsed)


  [4/20] proc-0206  parsed=True  ranked=['T0865', 'T0863', 'T0817']...  (40s elapsed)


...

  [16/20] proc-0018  parsed=True  ranked=['T0892', 'T0826', 'T0809']...  (83s elapsed)


  [17/20] proc-0015  parsed=True  ranked=['T1691', 'T0813', 'T0826']...  (86s elapsed)


  [18/20] proc-0150  parsed=True  ranked=['T0801', 'T0871', 'T0845']...  (90s elapsed)


  [19/20] proc-0079  parsed=True  ranked=['T0809', 'T0826', 'T0879']...  (94s elapsed)


  [20/20] proc-0201  parsed=True  ranked=['T0871', 'T0843', 'T0889']...  (97s elapsed)

Pilot done. parse failures 0/20  (97s total)

Worked example raw reply (first pilot item, truncated to 800 chars):
{"predicted_techniques": [{"technique_id": "T0862", "rank": 1, "evidence_for": "'trojanized installers planted on compromised vendor sites' directly matches manipulation of software distribution mechanisms and compromise/replacement of legitimate software on third-party or vendor websites; mirrors the Havex trojanized-software example cited in the technique description", "evidence_against": "none", "verdict": "supported"}, {"technique_id": "T0863", "rank": 2, "evidence_for": "'trojanized installers' implies a user must install/run the application for malicious code to execute, which is the user interaction this technique describes", "evidence_against": "the behaviour emphasizes the delivery channel (compromised vendor sites) rather than user interaction itself; no mention of spearphishi

## 5. Scaled run (n=50) — the real E-vs-D comparison

Method D is recomputed on this exact same sample for a fair, same-sample comparison.

In [6]:
sample = eval_data.sample_by_technique(examples, EVAL_N, seed=SAMPLE_SEED)
sample_ids = {ex.id for ex in sample}
covered = {tid for ex in sample for tid in ex.true_technique_ids}
print(f"Sampled {len(sample)} examples covering {len(covered)}/{len(valid_ids)} techniques.")
print("Sample by slice:", dict(Counter(ex.slice for ex in sample)))

method_d_results = []
method_e_results = []
t0 = time.time()
for i, ex in enumerate(sample, 1):
    candidates = method_d_candidates(ex.behavior_text)
    candidate_ids = {c[0] for c in candidates}

    method_d_results.append(
        metrics.MethodResult(report_id=ex.id, true_ids=ex.true_technique_ids, ranked_ids=[c[0] for c in candidates])
    )

    prompt = ach_prompts.build_ach_prompt(ex.behavior_text, candidates)
    pred = opencode_client.classify(FRONTIER_MODEL, prompt, candidate_ids, cache=cache, timeout=180)
    method_e_results.append(
        metrics.MethodResult(
            report_id=ex.id, true_ids=ex.true_technique_ids, ranked_ids=pred.ranked_ids,
            parse_failure=pred.parse_failure, hallucinated_id_count=len(pred.hallucinated),
        )
    )
    if i % 10 == 0 or i == len(sample):
        print(f"  [{i}/{len(sample)}]  ({time.time()-t0:.0f}s elapsed)")

pf = sum(r.parse_failure for r in method_e_results)
print(f"\nScaled run done. Method E parse failures {pf}/{len(method_e_results)}  ({time.time()-t0:.0f}s total)")

Sampled 50 examples covering 50/79 techniques.
Sample by slice: {'procedure': 47, 'synthetic-hard': 2, 'real': 1}


  [10/50]  (34s elapsed)


  [20/50]  (70s elapsed)


  [30/50]  (108s elapsed)


  [40/50]  (155s elapsed)


  [50/50]  (1652s elapsed)

Scaled run done. Method E parse failures 2/50  (1652s total)


## 6. Results — Method D vs Method E, same sample

In [7]:
rank_results = {
    "D: Retrieve+rerank (this sample)": method_d_results,
    "E: ACH arbiter (this sample)": method_e_results,
}
rank_scores = {name: metrics.rank_score_method(name, res) for name, res in rank_results.items()}

results_table = pd.DataFrame(
    [
        {
            "method": name,
            "MRR": round(s.mrr, 3),
            "MAP": round(s.map, 3),
            "Recall@1": round(s.recall_at_k[1], 3),
            "Recall@3": round(s.recall_at_k[3], 3),
            "n": s.n_reports,
        }
        for name, s in rank_scores.items()
    ]
).set_index("method")
results_table

,MRR,MAP,Recall@1,Recall@3,n
method,,,,,
D: Retrieve+rerank (this sample),0.508,0.51,0.42,0.58,50
E: ACH arbiter (this sample),0.640,0.64,0.61,0.66,50


In [8]:
parse_failures = sum(r.parse_failure for r in method_e_results)
hallucinated = sum(r.hallucinated_id_count for r in method_e_results)
print(f"Method E: {parse_failures}/{len(method_e_results)} parse failures, "
      f"{hallucinated} hallucinated IDs (an ID outside the offered {ARBITER_K} candidates) across the run.")

Method E: 2/50 parse failures, 0 hallucinated IDs (an ID outside the offered 6 candidates) across the run.


## 7. Decision

In [9]:
best = max(rank_scores.items(), key=lambda kv: kv[1].mrr)
ranked = sorted(rank_scores.items(), key=lambda kv: kv[1].mrr, reverse=True)
summary = " > ".join(f"{name} (MRR={s.mrr:.2f})" for name, s in ranked)

print("DRAFT summary (edit the markdown cell below with the real numbers once run):\n")
print(
    f"On a stratified sample of {EVAL_N} examples, same-sample D vs E:\n  {summary}.\n\n"
    f"Best: '{best[0]}' (MRR={best[1].mrr:.2f}, MAP={best[1].map:.2f}, "
    f"Recall@3={best[1].recall_at_k[3]:.2f}).\n"
    f"Method E parse-failure rate: {parse_failures}/{len(method_e_results)}."
)

DRAFT summary (edit the markdown cell below with the real numbers once run):

On a stratified sample of 50 examples, same-sample D vs E:
  E: ACH arbiter (this sample) (MRR=0.64) > D: Retrieve+rerank (this sample) (MRR=0.51).

Best: 'E: ACH arbiter (this sample)' (MRR=0.64, MAP=0.64, Recall@3=0.66).
Method E parse-failure rate: 2/50.


_(Fill in after the run: does the ACH arbiter beat, tie, or lose to the production
reranker on this sample? If it loses or ties, that is itself a real, useful finding —
it means Method D's current zero-LLM design is already close to what a frontier model
adds, which is a stronger, more defensible basis for the Phase 2 production-alignment
decisions than assuming a frontier LLM must help.)_